# L12 · 空间变换机器 — 矩阵乘法

**目标**：矩阵乘向量 = 对向量做旋转/拉伸；矩阵乘矩阵 = 把多个变换串起来。

**为什么对 Aurora 重要**：mel 滤波矩阵 `(M, N)` 把 FFT 频谱 `(N,)` 压缩成 mel 频谱 `(M,)`，调用的就是一次 `matvec`；DFT 的朴素实现 `dft()` 同样是 N×N 复数矩阵乘信号向量。

## 本课剧情：启动空间变换机器

矩阵不是数字表格，而是一个线性变换：`A @ x` 把形状 `(n,)` 的向量映射到 `(m,)` 的空间，矩阵的 shape `(m, n)` 直接说明了输入和输出维度。这节课先用缩放矩阵和旋转矩阵建立直觉，然后手写 `matvec` 展开乘法的循环结构。

## 1. 矩阵就是二维数组；矩阵 × 向量

矩阵 `A` 的 shape `(m, n)` 直接告诉你：输入必须是长度 `n` 的向量，输出是长度 `m` 的向量。`A @ x` 的第 `i` 个元素等于矩阵第 `i` 行与 `x` 的点积：Σⱼ A[i,j]·x[j]。

这是"m 个点积"的视角：矩阵的每一行是一组权重，和输入做点积，产生输出的一个维度。mel 滤波矩阵（`src/aurora/audio/mel.py` 的 `mel_filterbank()` 返回值，shape `(M, N)`）的每一行是一个三角滤波器，和 FFT 频谱做点积，得到该 mel 频段的能量——这就是把 `(N,)` 频谱压缩成 `(M,)` mel 频谱的本质。

## 符号入口：先看形状，再看运算

线性代数里的对象都有明确形状：向量是 `(n,)`，矩阵是 `(m, n)`，矩阵乘向量把 `(n,)` 变成 `(m,)`。每个例子先标出输入维度和输出维度，再看具体数值。

In [ ]:
import numpy as np
W = np.array([[2.0, 0.0],
              [0.0, 3.0]])   # 把 x 拉伸 2 倍、y 拉伸 3 倍
x = np.array([1.0, 1.0])
print('W @ x =', W @ x)     # [2. 3.]

## 动手观察：线代对象先看形状，再看意义

运行下面的代码，注意 `A @ v` 前后的 shape：输入 `v` 是 `(2,)`，输出还是 `(2,)`，但矩阵 `A` 的对角元素决定了每个分量被拉伸的倍数。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：遍历几个向量，观察矩阵如何改变它们

把几个方向不同的向量送进同一个矩阵，观察输出向量的方向和长度如何随输入方向变化。

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


## 2. 旋转矩阵：把向量转一个角度

In [ ]:
import matplotlib.pyplot as plt
theta = np.pi/4  # 45°
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
x = np.array([3.0, 0.0]); xr = R @ x
plt.figure(figsize=(4,4))
plt.quiver(0,0,x[0],x[1], angles='xy',scale_units='xy',scale=1,color='gray')
plt.quiver(0,0,xr[0],xr[1],angles='xy',scale_units='xy',scale=1,color='red')
plt.xlim(-1,4);plt.ylim(-1,4);plt.grid(True,alpha=.3)
plt.gca().set_aspect('equal');plt.title('rotation by 45°');plt.show()

## 3. ✏️ 你的任务：手写矩阵×向量 `matvec`

不准用 `@`，用循环展开它的计算过程。

**推理路线**：
1. 输出长度 = 矩阵行数 `m`，先建零数组：`out = np.zeros(A.shape[0])`
2. 对每个行索引 `i`，计算第 `i` 行与 `x` 的点积，存入 `out[i]`
3. 用 `np.dot(A[i], x)` 完成单行点积，等价于 Σⱼ A[i,j]·x[j]，无需写内层循环

**参考输入输出**：`A=[[1,0],[0,2]], x=[3,4]` → `[1×3+0×4, 0×3+2×4] = [3, 8]`

<details><summary>点击查看参考实现</summary>

```python
def matvec(A, x):
    out = np.zeros(A.shape[0])
    for i in range(A.shape[0]):
        out[i] = np.dot(A[i], x)
    return out
```

</details>

### 写代码前，先把变量表补完整

写 `matvec` 前明确三件事：
- 输入：矩阵 `A`（形状 `(m, n)`）和向量 `x`（形状 `(n,)`）
- 关键步骤：对每个行索引 `i`，计算 `np.dot(A[i], x)` 得到输出的第 `i` 个元素
- 返回：形状 `(m,)` 的向量，与 `A @ x` 数值相同

In [ ]:
def matvec(A, x):
    # ✏️ TODO: 用循环返回 A @ x（每个元素是一行与 x 的点积）
    out = np.zeros(A.shape[0])
    # for i in range(A.shape[0]):
    #     out[i] = ...
    return out


In [ ]:
A = np.array([[1.0,2.0],[3.0,4.0]])
x = np.array([5.0,6.0])
print('我的 matvec:', matvec(A, x))
print('numpy  A@x :', A @ x)
assert np.allclose(matvec(A, x), A @ x), '应与 A@x 一致'
print('\n✅ 通过：矩阵乘向量 = 一组点积。')


**🔗 Aurora 连接**：`src/aurora/audio/mel.py` 的 `mel_filterbank()` 返回 shape `(M, N)` 的矩阵，`mel_spectrogram()` 用 `fb @ magnitude` 把 FFT 幅度谱压成 mel 频谱；`src/aurora/audio/transforms.py` 的 `dft()` 核心是 `twiddle @ x`，N×N 复数矩阵乘信号向量——两处都是 `matvec` 的直接实例。

## 🎨 图示：矩阵×向量 = 列的线性组合

In [ ]:
from laviz import style, mat_times_vec
style()
mat_times_vec([[1,2],[3,4],[5,6]],[2,1]);

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：对角矩阵与全1方阵

构造对角矩阵 `np.diag([2,3])`，输入 `[1,1]`，确认输出是 `[2,3]`——行 0 的权重 `[2,0]` 只取 x[0]，行 1 的权重 `[0,3]` 只取 x[1]，各维度独立缩放。再把矩阵换成全1方阵 `np.ones((2,2))`，输入同样的 `[1,1]`，观察输出变成 `[2,2]`——每一行 `[1,1]` 与 `[1,1]` 的点积 = 1+1 = 2，输出是输入所有元素之和。

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 本课收束

现在可以用 `matvec(A, x)` 手算矩阵乘法，并与 `A @ x` 逐元素对比验证。这个操作在 Aurora 里对应 mel 滤波：`(M, N)` 矩阵作用于 FFT 输出 `(N,)`，得到 mel 频谱 `(M,)`。下一节看特殊矩阵，正交矩阵和对角矩阵是理解特征分解的基础。

In [ ]:
# 小检查：先看 shape，再做运算
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])
A = np.array([[1.0, 2.0], [0.0, 1.0]])

print('a.shape =', a.shape)
print('b.shape =', b.shape)
print('A.shape =', A.shape)
print('a + b =', a + b)
print('A @ a =', A @ a)
print('a dot b =', float(a @ b))
